# Interpretation

SHAP values from the fitted LightGBM — which features move risk and in which direction. A random sample drives the beeswarm; global importance is the mean absolute SHAP.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from src.data import load_selected

INTERIM = Path("../data/interim")
X, y = load_selected()
bp = json.loads((INTERIM / "best_params.json").read_text()) if (INTERIM / "best_params.json").exists() else {}
X.shape

: 

## Fit

In [ ]:
p = {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 50,
     "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("lgb", {})}
model = lgb.LGBMClassifier(**p, subsample_freq=1, n_jobs=-1, verbose=-1).fit(X, y)

## SHAP summary

In [ ]:
samp = X.sample(5000, random_state=42)
sv = model.booster_.predict(samp, pred_contrib=True)[:, :-1]
shap.summary_plot(sv, samp, max_display=20, show=True)

## Global importance

In [ ]:
imp = pd.Series(np.abs(sv).mean(0), index=samp.columns).sort_values(ascending=False)
imp.head(20)

## Top feature effect

In [ ]:
shap.dependence_plot(imp.index[0], sv, samp, show=True)